# Code-Gen Pipeline — Prompt Inspector

Compile and inspect the assembled prompt for every code-gen stage.

| Stage | What it builds |
|---|---|
| state1 | Entity list (JSON) |
| state1b | Policy outline (JSON) |
| state1c | Entity dependency DAG (JSON) |
| state1d | Metrics draft (JSON) |
| state2 | Entity class code (Python) |
| state3 | Environment class code (Python) |
| state4 | Policy class code (Python) |
| state5 | Policy judge (LLM-as-judge) |

In [1]:
import sys, pathlib, json, importlib.util
from IPython.display import display, Markdown

# ── path setup ──────────────────────────────────────────────────────────────
REPO    = pathlib.Path.cwd().parents[1]          # Garbage-Management-Simulation-Engine/
BACKEND = REPO / "engine" / "backend"
_SVC    = BACKEND / "app" / "services"

# Direct module load — avoids triggering app/__init__.py → FastAPI import
def _load_module(name: str, path: pathlib.Path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

P = _load_module("code_gen_prompts", _SVC / "code_gen_prompts.py")

# ── display helper ──────────────────────────────────────────────────────────
def show(title: str, text: str, lang: str = "text", max_lines: int = 0) -> None:
    lines = text.splitlines()
    if max_lines and len(lines) > max_lines:
        body = "\n".join(lines[:max_lines]) + f"\n\n… ({len(lines) - max_lines} more lines)"
    else:
        body = text
    display(Markdown(f"### {title}\n\n```{lang}\n{body}\n```"))

def show_json(title: str, obj, max_lines: int = 0) -> None:
    show(title, json.dumps(obj, indent=2, ensure_ascii=False), lang="json", max_lines=max_lines)

print("Setup OK — backend path:", BACKEND)

Setup OK — backend path: /Users/rapepong/Development/kmutt/Garbage-Management-Simulation-Engine/engine/backend


---
## Fixtures
Minimal stub data used to assemble every prompt below.  
Replace with real causal data to see production-equivalent output.

In [2]:
CAUSAL_DATA = json.dumps([
    {
        "sentence": "Waste bins are placed throughout the building. When a bin is full, a janitor empties it into a collection truck.",
        "classes": [
            {"id": "waste_bin", "label": "Waste Bin", "sentence_type": "CS"},
            {"id": "janitor",   "label": "Janitor",   "sentence_type": "CS"},
            {"id": "truck",     "label": "Collection Truck", "sentence_type": "CS"},
        ]
    }
], ensure_ascii=False)

ENTITIES = [
    {"id": "waste_bin", "label": "Waste Bin",        "type": "resource", "frequency": 5},
    {"id": "janitor",   "label": "Janitor",           "type": "actor",    "frequency": 3},
    {"id": "truck",     "label": "Collection Truck",  "type": "resource", "frequency": 2},
]

POLICIES = [
    {
        "rule_id": "empty_full_bin",
        "label": "Empty Full Bin",
        "trigger": "waste_bin.fill_level >= capacity",
        "target_entity_id": "waste_bin",
        "target_method": "empty",
        "inputs": ["janitor"],
        "description": "Janitor empties a full waste bin into the truck.",
    }
]

EDGES = [
    {"from": "janitor", "to": "waste_bin", "reason": "janitor calls waste_bin.empty()"},
    {"from": "janitor", "to": "truck",     "reason": "janitor deposits waste into truck"},
]

METRICS = [
    {
        "name": "bin_fill_level",
        "label": "Bin Fill Level",
        "unit": "%",
        "agg": "mean",
        "viz": "line",
        "entity_id": "waste_bin",
        "expected_variable": "fill_level",
        "chart_type": "line",
        "how_to_interpret": "Average fill level across all bins per tick.",
        "required_attrs": [{"entity": "waste_bin", "attr": "fill_level"}],
        "sampling_event": "tick",
        "rationale": "Core operational metric for overflow prevention.",
    }
]

SAMPLE_ENTITY_OBJ = ENTITIES[0]   # waste_bin — used in state2 demo
SAMPLE_RULE       = POLICIES[0]   # empty_full_bin — used in state4 demo

print("Fixtures ready.")

Fixtures ready.


---
## Code Templates
Raw base-class files injected into State 2 / 3 / 4 prompts.

In [3]:
TEMPLATE_DIR = BACKEND / "app" / "services" / "templates"

TMPL_ENTITY  = (TEMPLATE_DIR / "entity_object_template.py").read_text()
TMPL_ENV     = (TEMPLATE_DIR / "environment_template.py").read_text()
TMPL_POLICY  = (TEMPLATE_DIR / "policy_template.py").read_text()

templates = {
    "entity_object_template.py":  TMPL_ENTITY,
    "environment_template.py":    TMPL_ENV,
    "policy_template.py":         TMPL_POLICY,
}

for name, src in templates.items():
    show(name, src, lang="python", max_lines=40)

### entity_object_template.py

```python
from abc import ABC
from typing import TYPE_CHECKING, Optional, List

if TYPE_CHECKING:
    from .environment_template import SimulationEnvironment
    from .policy_template import Policy

# ---------------------------------------------------------
# UNIFIED entity_object TEMPLATE
# ---------------------------------------------------------
class entity_object(ABC):
    """
    Unified template for all entity_objects in the simulation.

    entity_objects can have active traits (perceive, decide, act) and/or passive traits (hold state, be acted upon).

    ACTIVE TRAIT: Override perceive(), decide_action(), and act() methods
    - Used for entity_objects that can autonomously perceive, make decisions, and take actions
    - Examples: Janitor, Student, Truck Driver

    PASSIVE TRAIT: Override get_status() and on_interact() methods
    - Used for entity_objects that hold state and can be acted upon by other entity_objects
    - Examples: Garbage Bin, Waste Buffer, Large Waste Item

    HYBRID (Both Traits): Override all five methods above
    - Used for entity_objects that can both act autonomously AND be acted upon
    - Examples: Smart Garbage Truck (acts autonomously but also has capacity state),
                Sorting Station (processes waste but also accumulates it)

    The trait(s) an entity_object has are determined by which methods you implement.
    Only implement the methods relevant to your entity_object's capabilities.

    POLICIES: entity_objects can have policies attached that modify their behavior.
    """

    def __init__(self, entity_object_id: str):
        self.entity_object_id = entity_object_id
        self.state = "Idle"
        self._policies: List['Policy'] = []  # Policies attached to this entity_object


… (161 more lines)
```

### environment_template.py

```python
import json
import pathlib
from typing import Any, Dict, List, Optional, TYPE_CHECKING

if TYPE_CHECKING:
    from .entity_object_template import entity_object
    from .policy_template import Policy

# ---------------------------------------------------------
# THE MEDIATOR (Environment)
# ---------------------------------------------------------
class SimulationEnvironment:
    """
    Acts as the Mediator. All entities can talk to the environment.

    The environment maintains:
    1. entity_objects (both active and passive)
    2. Global properties that affect all entity_objects
    3. Global policies and system behaviors
    4. Map graph loaded from map.json (nodes + edges + adjacency)

    Active entity_objects can modify global properties through set_property().
    All entity_objects can observe global properties through get_property().
    """
    def __init__(self, entities: list = None, policies: list = None):
        self.entities: List['entity_object'] = list(entities or [])
        self.entity_objects: List['entity_object'] = self.entities  # alias
        self.policies: List['Policy'] = list(policies or [])
        self.behaviors: list = []

        # Global properties that affect all entity_objects
        self._global_properties: Dict[str, Any] = {}

        # Map graph loaded from map.json
        self._nodes: Dict[str, dict] = {}
        self._edges: List[dict] = []
        self._adjacency: Dict[str, List[str]] = {}
        self._load_map()

        # Backward compatibility

… (104 more lines)
```

### policy_template.py

```python
from abc import ABC, abstractmethod
from typing import Any, TYPE_CHECKING, Optional

if TYPE_CHECKING:
    from .environment_template import SimulationEnvironment
    from .entity_object_template import entity_object

# ---------------------------------------------------------
# POLICY TEMPLATE (Strategy Pattern)
# ---------------------------------------------------------
class Policy(ABC):
    """
    Template for Policies that can be applied to entity_objects.
    Policies define behaviors or rules that entity_objects follow (Strategy Pattern).

    Examples:
    - WasteRevenuePolicy: Determines how waste collection revenue is divided
    - CollectionSchedulePolicy: Defines when and how entity_objects collect waste
    - CapacityManagementPolicy: Rules for handling entity_object capacity limits
    """

    @property
    @abstractmethod
    def policy_name(self) -> str:
        """Name identifier for this policy."""
        pass

    def before_tick(self, env: 'SimulationEnvironment', dt: float) -> None:
        """Hook called before entities step each tick. Override with real logic."""
        pass

    def after_tick(self, env: 'SimulationEnvironment', dt: float) -> None:
        """Hook called after entities step each tick. Override with real logic."""
        pass

    @abstractmethod
    def apply(self, entity_object: 'entity_object', context: dict, env: Optional['SimulationEnvironment'] = None) -> dict:
        """
        Applies the policy to a specific entity_object with given context.


… (22 more lines)
```

---
## State 1 — Entity List
**Input**: causal data  
**Output schema**: `{entities: [{id, label, type, frequency}]}`

In [4]:
prompt_s1, schema_s1 = P.build_state1_entity_list_prompt(CAUSAL_DATA)

show_json("State 1 — Response Schema", schema_s1)
show("State 1 — Assembled Prompt", prompt_s1)

### State 1 — Response Schema

```json
{
  "type": "object",
  "properties": {
    "entities": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "id": {
            "type": "string"
          },
          "label": {
            "type": "string"
          },
          "type": {
            "type": "string",
            "enum": [
              "actor",
              "resource",
              "environment",
              "policy"
            ]
          },
          "frequency": {
            "type": "integer"
          }
        },
        "required": [
          "id",
          "label",
          "type"
        ]
      }
    },
    "warning": {
      "type": "string"
    }
  },
  "required": [
    "entities"
  ]
}
```

### State 1 — Assembled Prompt

```text
Analyze the provided causal data and extract all distinct entities (actors, resources, environments, policies). Count how many times each entity is mentioned across the full causal data to determine frequency and generation priority. Apply dedup and exclusion policies. Return JSON only using the configured entity list schema.

Do NOT include a 'priority' field. The caller computes priority client-side.

Use this JSON schema exactly:
{
  "entities": [
    {"id": "string", "label": "string", "type": "string", "frequency": 0}
  ]
}
Field rules:
- id: normalized snake_case identifier derived from label.
- label: exact surface form as it appears in causal data.
- type: one of ["actor", "resource", "environment", "policy"].
- frequency: integer count of occurrences across all causal data.
Do NOT emit a 'priority' field — ranking is done by the caller, not the model.

Deduplicate entities by semantic identity. If two labels in causal data refer to the same real-world actor or resource (e.g. 'forklift' and 'Forklift Operator'), keep one canonical entity. Prefer the most frequent label as the canonical id. Merge frequency counts. Do not create separate entities for plural vs singular forms of the same concept.

Exclude abstract, placeholder, or meta-level terms from the entity list. Do not emit entities for: generic system concepts (e.g. 'system', 'process', 'data'), causal connectors (e.g. 'cause', 'effect', 'result'), or any label the causal data explicitly marks as deprecated, removed, or inactive.

Minimize output tokens: return only required fields and code. No explanations, no duplicate class definitions, no markdown wrappers unless explicitly requested.

Causal data:
[{"sentence": "Waste bins are placed throughout the building. When a bin is full, a janitor empties it into a collection truck.", "classes": [{"id": "waste_bin", "label": "Waste Bin", "sentence_type": "CS"}, {"id": "janitor", "label": "Janitor", "sentence_type": "CS"}, {"id": "truck", "label": "Collection Truck", "sentence_type": "CS"}]}]
```

---
## State 1b — Policy Outline
**Input**: causal data + entity list  
**Output schema**: `{policies: [{rule_id, trigger, target_entity_id, target_method, ...}]}`

In [5]:
prompt_s1b, schema_s1b = P.build_state1b_policy_outline_prompt(CAUSAL_DATA, ENTITIES)

show_json("State 1b — Response Schema", schema_s1b)
show("State 1b — Assembled Prompt", prompt_s1b)

### State 1b — Response Schema

```json
{
  "type": "object",
  "properties": {
    "policies": {
      "type": "array",
      "items": {
        "type": "object",
        "properties": {
          "rule_id": {
            "type": "string"
          },
          "label": {
            "type": "string"
          },
          "trigger": {
            "type": "string"
          },
          "target_entity_id": {
            "type": "string"
          },
          "target_method": {
            "type": "string"
          },
          "inputs": {
            "type": "array",
            "items": {
              "type": "string"
            }
          },
          "description": {
            "type": "string"
          }
        },
        "required": [
          "rule_id",
          "trigger",
          "target_entity_id",
          "target_method"
        ]
      }
    }
  },
  "required": [
    "policies"
  ]
}
```

### State 1b — Assembled Prompt

```text
Extract EVERY causal mechanism that the system exhibits, as evidenced by the causal data. A causal mechanism is any condition-action pair — a condition the system currently responds to and the action it currently takes. Include all types: enforcement rules, state transitions, feedback responses, resource reactions, overflow/underflow handlers, and any other behavior the causal data shows. Do NOT limit the number of policies — emit one entry per distinct causal mechanism, however many that is. Do NOT propose improvements, corrections, or new behaviors. Describe the system as-is: what condition already triggers it, which entity already handles it, and what method that entity already performs. For each mechanism, output one entry: the observed trigger, the target entity (or 'environment'), and a method name that reads as the entity's own behavior (not a policy label).

Use this JSON schema exactly:
{
  "policies": [
    {
      "rule_id": "string (snake_case)",
      "label": "string (human-readable rule name)",
      "trigger": "string (when the rule fires — condition or event from causal data)",
      "target_entity_id": "string (entity.id from State 1 — the class the policy will mutate)",
      "target_method": "string (snake_case method name the policy will call on the target entity)",
      "inputs": ["string", ...],
      "description": "string (one sentence)"
    }
  ]
}
Field rules:
- target_entity_id MUST match an entity.id from the provided entity list. If a rule mutates the
  environment, use the literal value 'environment'.
- target_method MUST be the public method name the entity (or environment) is expected to
  expose. Stage 2 will be told to define this method on the target class so the policy has a
  stable interface to call.
- Do NOT emit policy bodies. This stage is a contract preview only.

Minimize output tokens: return only required fields and code. No explanations, no duplicate class definitions, no markdown wrappers unless explicitly requested.

Entity list (output of State 1):
{"entities": [{"id": "waste_bin", "label": "Waste Bin", "type": "resource", "frequency": 5}, {"id": "janitor", "label": "Janitor", "type": "actor", "frequency": 3}, {"id": "truck", "label": "Collection Truck", "type": "resource", "frequency": 2}]}

Causal data:
[{"sentence": "Waste bins are placed throughout the building. When a bin is full, a janitor empties it into a collection truck.", "classes": [{"id": "waste_bin", "label": "Waste Bin", "sentence_type": "CS"}, {"id": "janitor", "label": "Janitor", "sentence_type": "CS"}, {"id": "truck", "label": "Collection Truck", "sentence_type": "CS"}]}]
```

### test output

In [6]:
policies = [
{
"rule_id": "bma_disposal_routing",
"label": "BMA Waste Disposal Routing",
"trigger": "Waste arrives at the sorting facility",
"target_entity_id": "entity-27-bma",
"target_method": "receive_disposal_waste",
"inputs": ["waste_volume"],
"description": "Directs waste from the sorting facility to the Bangkok Metropolitan Administration for final disposal."
},
{
"rule_id": "building_waste_aggregation",
"label": "Building Waste Aggregation",
"trigger": "Units or offices generate waste",
"target_entity_id": "entity-53-faculties",
"target_method": "aggregate_waste_at_designated_spots",
"inputs": ["unit_id", "building_location"],
"description": "Faculties and offices consolidate waste at specific designated locations under or near their respective buildings."
},
{
"rule_id": "waste_ownership_labeling",
"label": "Waste Ownership Labeling",
"trigger": "Waste is placed at designated spots",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "apply_ownership_label",
"inputs": ["unit_identifier"],
"description": "Applies identifying labels to waste piles to track which unit or department generated the waste."
},
{
"rule_id": "finance_payment_trigger",
"label": "Finance Payment Trigger",
"trigger": "Waste data is submitted to the Finance Division",
"target_entity_id": "entity-53-faculties",
"target_method": "receive_payment_allocation",
"inputs": ["waste_data"],
"description": "Triggers the transfer of funds to specific units based on the waste data provided to the Finance Division."
},
{
"rule_id": "waste_revenue_allocation",
"label": "Waste Revenue Split",
"trigger": "Revenue is generated from unit waste",
"target_entity_id": "entity-53-faculties",
"target_method": "allocate_revenue_split",
"inputs": ["total_revenue"],
"description": "Allocates 80% of waste revenue to the generating unit and 20% to the University."
},
{
"rule_id": "rdf_fuel_conversion",
"label": "RDF Fuel Conversion",
"trigger": "General waste is separated and cleaned",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "convert_to_fuel_waste",
"inputs": ["waste_type"],
"description": "Processes cleaned general waste into Refuse Derived Fuel (RDF) for delivery to N15."
},
{
"rule_id": "infectious_waste_segregation",
"label": "Infectious Waste Segregation",
"trigger": "COVID-19 pandemic conditions are active",
"target_entity_id": "environment",
"target_method": "deploy_infectious_bins",
"inputs": ["mask_volume", "atk_kit_volume"],
"description": "Triggers the deployment of specialized bins specifically for infectious materials like masks and test kits."
},
{
"rule_id": "ewaste_hour_exchange",
"label": "E-Waste Hour Exchange",
"trigger": "Student donates electronic waste",
"target_entity_id": "entity-20-students",
"target_method": "receive_activity_hours",
"inputs": ["ewaste_quantity"],
"description": "Awards activity hours to students who donate electronic waste at the designated center."
},
{
"rule_id": "reusable_part_donation",
"label": "Reusable Part Donation",
"trigger": "E-waste is identified as functional or containing spare parts",
"target_entity_id": "entity-14-mirror-foundation",
"target_method": "accept_reusable_electronics",
"inputs": ["component_data"],
"description": "Routes reusable electronic components to the Mirror Foundation for repairing rural computers."
},
{
"rule_id": "organic_fertilizer_processing",
"label": "Organic Fertilizer Processing",
"trigger": "Tree debris is collected",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "process_into_fertilizer",
"inputs": ["debris_volume"],
"description": "Converts tree and organic plant debris into fertilizer."
},
{
"rule_id": "housekeeper_sorting_trigger",
"label": "Housekeeper Sorting Trigger",
"trigger": "Students or staff discard waste in building bins",
"target_entity_id": "entity-2-housekeepers",
"target_method": "perform_preliminary_sorting",
"inputs": ["bin_load"],
"description": "Triggers housekeepers to perform a first level of waste sorting based on what was discarded."
},
{
"rule_id": "waste_cage_transfer",
"label": "Waste Cage Transfer",
"trigger": "Internal bins are weighed and collected",
"target_entity_id": "entity-2-housekeepers",
"target_method": "move_to_holding_cage",
"inputs": ["waste_weight"],
"description": "Directs housekeepers to move weighed waste from internal building points to external holding cages."
},
{
"rule_id": "external_garden_collection",
"label": "External Garden Collection",
"trigger": "Waste exists in external campus areas",
"target_entity_id": "entity-145-waste-collectors",
"target_method": "collect_external_waste",
"inputs": ["route_plan"],
"description": "Triggers the garden team to drive trucks and collect waste from external campus perimeters."
},
{
"rule_id": "sorting_plant_delivery",
"label": "Sorting Plant Delivery",
"trigger": "Waste collection rounds are complete",
"target_entity_id": "entity-145-waste-collectors",
"target_method": "transport_to_sorting_plant",
"inputs": ["total_load"],
"description": "Initiates the transport of all collected campus waste to the centralized sorting plant."
},
{
"rule_id": "manual_sorting_segregation",
"label": "Manual Sorting Segregation",
"trigger": "Waste is identified as manually sortable",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "pile_for_recycling",
"inputs": ["material_type"],
"description": "Separates sortable recyclable materials into distinct piles at the facility."
},
{
"rule_id": "non_recyclable_compression",
"label": "Non-Recyclable Compression",
"trigger": "Waste is identified as unsortable",
"target_entity_id": "entity-27-bma",
"target_method": "compress_in_yellow_bins",
"inputs": ["unsortable_volume"],
"description": "Directs non-recyclable waste into yellow BMA compression bins for removal."
},
{
"rule_id": "sorting_failure_mechanism",
"label": "Sorting Capacity Failure",
"trigger": "Sorter headcount is insufficient for waste volume",
"target_entity_id": "environment",
"target_method": "fail_sorting_operation",
"inputs": ["sorter_count", "waste_load"],
"description": "Causes a breakdown in sorting effectiveness when only 3-4 people handle the entire campus volume."
},
{
"rule_id": "morning_overflow_trigger",
"label": "Morning Overflow Trigger",
"trigger": "Housekeepers compress morning rounds into holding points",
"target_entity_id": "environment",
"target_method": "trigger_point_overflow",
"inputs": ["morning_volume"],
"description": "Causes holding points to overflow specifically during morning hours due to the intensity of early collections."
},
{
"rule_id": "late_disposal_residue",
"label": "Late Disposal Residue",
"trigger": "Waste is brought down after the 3:00 PM collection truck",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "remain_as_residual_overnight",
"inputs": ["disposal_time"],
"description": "Ensures waste remains as a residual overnight if it misses the final daily pickup window."
},
{
"rule_id": "overflow_complaint_feedback",
"label": "Overflow Complaint Feedback",
"trigger": "Waste overflow is visible at collection points",
"target_entity_id": "entity-19-staff",
"target_method": "report_overflow_issue",
"inputs": ["location_id"],
"description": "Triggers staff or students to file complaints when waste accumulation exceeds bin capacity."
},
{
"rule_id": "management_warning_response",
"label": "Management Warning Response",
"trigger": "Waste overflow exceeds tolerance",
"target_entity_id": "entity-145-waste-collectors",
"target_method": "accelerate_waste_clearance",
"inputs": ["warning_level"],
"description": "Triggers the waste team to quickly manage overflow following a warning from the working committee."
},
{
"rule_id": "construction_waste_deposit",
"label": "Construction Waste Deposit",
"trigger": "Iron, cement, or large debris is generated",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "deposit_in_gray_bins",
"inputs": ["material_size"],
"description": "Routes heavy construction materials like iron and cement into specialized gray bins."
},
{
"rule_id": "mixed_transport_protocol",
"label": "Mixed Transport Protocol",
"trigger": "Initial waste loading for transport occurs",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "mix_types_for_transit",
"inputs": ["diversified_sources"],
"description": "Mixes different waste types during the initial collection phase for later separation at the destination."
},
{
"rule_id": "large_waste_scheduling",
"label": "Large Waste Scheduling",
"trigger": "Large construction waste is reported",
"target_entity_id": "entity-145-waste-collectors",
"target_method": "schedule_oversize_pickup",
"inputs": ["item_dimensions"],
"description": "Triggers the scheduling of specific pickup rounds for oversized materials that cannot fit standard rounds."
},
{
"rule_id": "unaffiliated_waste_impact",
"label": "Unaffiliated Waste Impact",
"trigger": "Students without clear departmental affiliation discard waste",
"target_entity_id": "environment",
"target_method": "increase_management_unpredictability",
"inputs": ["student_population"],
"description": "Increases the difficulty of waste management due to lack of departmental accountability for student-generated waste."
},
{
"rule_id": "special_pickup_request",
"label": "Special Pickup Request",
"trigger": "Recyclable volume exceeds standard capacity",
"target_entity_id": "entity-145-waste-collectors",
"target_method": "coordinate_unscheduled_pickup",
"inputs": ["waste_volume_est"],
"description": "Triggers coordination of unscheduled pickups when volume is too high for regular monthly rounds."
},
{
"rule_id": "event_waste_generation",
"label": "Event Waste Generation",
"trigger": "Sports or student activities occur",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "accumulate_event_debris",
"inputs": ["event_type"],
"description": "Triggers the sudden generation of large quantities of wood, plywood, and general waste from campus events."
},
{
"rule_id": "external_landfill_routing",
"label": "External Landfill Routing",
"trigger": "Waste is identified as unusable construction debris",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "route_to_external_landfill",
"inputs": ["usability_flag"],
"description": "Directs unusable construction waste to an external landfill near Wat Bua Phan."
},
{
"rule_id": "shop_floor_storage",
"label": "Shop Floor Storage",
"trigger": "Faculties have internal workshop space",
"target_entity_id": "entity-53-faculties",
"target_method": "store_internally_until_pickup",
"inputs": ["shop_capacity"],
"description": "Allows faculties with workshops to store waste internally until collectors arrive at the shop floor."
},
{
"rule_id": "monthly_building_piling",
"label": "Monthly Building Piling",
"trigger": "Monthly recycling round schedule is active",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "pile_at_building_front",
"inputs": ["schedule_date"],
"description": "Causes recyclable waste to be piled in front of buildings early in the morning for monthly collection."
},
{
"rule_id": "shift_end_disposal",
"label": "Shift End Disposal",
"trigger": "Housekeepers reach shift end at 5:00 PM",
"target_entity_id": "entity-2-housekeepers",
"target_method": "perform_late_dump",
"inputs": ["shift_time"],
"description": "Triggers housekeepers to discard trash exactly at 5:00 PM, causing a mismatch with the earlier 4:00 PM collection cutoff."
},
{
"rule_id": "overtime_cost_incurrence",
"label": "Overtime Cost Incurrence",
"trigger": "Additional collection rounds are requested after 5:00 PM",
"target_entity_id": "environment",
"target_method": "charge_overtime_fee",
"inputs": ["round_count"],
"description": "Incurs financial penalties/costs for the university when waste collection must happen outside standard hours."
},
{
"rule_id": "pest_attraction_mechanism",
"label": "Pest Attraction Mechanism",
"trigger": "Food waste is stored overnight in office spaces",
"target_entity_id": "environment",
"target_method": "attract_rats_and_ants",
"inputs": ["food_waste_age"],
"description": "Triggers the presence of rats and ants when food waste remains in office environments."
},
{
"rule_id": "lab_safety_violation",
"label": "Lab Safety Violation",
"trigger": "Laboratory disposal regulations are ignored",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "spill_chemical_materials",
"inputs": ["non_compliance_event"],
"description": "Leads to chemical leaks and personal injury when lab waste protocols are not followed."
},
{
"rule_id": "weight_data_discrepancy",
"label": "Weight Data Discrepancy",
"trigger": "Personnel bypass weighing procedures",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "log_guessed_weight",
"inputs": ["personnel_mood"],
"description": "Causes extreme data inaccuracy as weight data is recorded based on guesses rather than actual measurements."
},
{
"rule_id": "design_constraint_impact",
"label": "Design Constraint Impact",
"trigger": "Waste cage design is limited by budget",
"target_entity_id": "entity-2-housekeepers",
"target_method": "experience_operational_difficulty",
"inputs": ["budget_limit"],
"description": "Forces cleaners to struggle with waste handling due to the too-small and restrictive size of the waste cages."
},
{
"rule_id": "equipment_damage_mechanism",
"label": "Equipment Damage Mechanism",
"trigger": "Garbage collectors use rough physical handling",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "break_bin_components",
"inputs": ["force_applied"],
"description": "Leads to the destruction and breaking of waste bins when collectors use excessive force during pickup."
},
{
"rule_id": "event_separation_assistance",
"label": "Event Separation Assistance",
"trigger": "Campus events are active",
"target_entity_id": "entity-20-students",
"target_method": "monitor_waste_separation",
"inputs": ["volunteer_count"],
"description": "Deploys student volunteers to stand by bins and guide people in separating waste during events."
},
{
"rule_id": "reusable_resource_reduction",
"label": "Reusable Resource Reduction",
"trigger": "Reusable glasses and plates are rented for events",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "reduce_waste_volume_by_mass",
"inputs": ["equipment_count"],
"description": "Reduces total campus waste by hundreds of kilograms by substituting disposables with rented reusable equipment."
},
{
"rule_id": "convenience_dumping_habit",
"label": "Convenience Dumping Habit",
"trigger": "Any bin is within immediate reach of a user",
"target_entity_id": "entity-canonical-0-waste",
"target_method": "receive_unsorted_waste",
"inputs": ["distance_to_bin"],
"description": "Causes users to ignore separation rules and throw waste into the nearest available bin for convenience."
}
]
len(policies)

40

---
## State 1c — Entity Dependencies
**Input**: causal data + entity list + policy outline  
**Output schema**: `{edges: [{from, to, reason}]}`

In [ ]:
prompt_s1c, schema_s1c = P.build_state1c_entity_dependencies_prompt(
    CAUSAL_DATA, ENTITIES, POLICIES
)

show_json("State 1c — Response Schema", schema_s1c)
show("State 1c — Assembled Prompt", prompt_s1c)

---
## State 1d — Metrics Draft
**Input**: entity list + policy outline + dependency edges  
**Output schema**: `{metrics: [{name, label, entity_id, expected_variable, ...}]}`

In [ ]:
prompt_s1d, schema_s1d = P.build_state1d_metrics_draft_prompt(
    entities=ENTITIES,
    policy_outline=POLICIES,
    dependency_edges=EDGES,
)

show_json("State 1d — Response Schema", schema_s1d)
show("State 1d — Assembled Prompt", prompt_s1d)

---
## State 2 — Entity Object Code
**Template injected**: `entity_object_template.py`  
**Iterative**: one call per entity in topological order  
**Output**: Python class subclassing `entity_object`

Two sub-cells:
1. Stable cached context (shared across all iterations)
2. Per-entity variable prompt (for one sample entity)

In [ ]:
# ── 2a: Cached context (sent once, reused for all entities) ─────────────────
cached_parts = P.build_state2_cached_context(causal_data=CAUSAL_DATA)
show(
    "State 2 — Cached Context (stable prefix, shared across all entity iterations)",
    "\n\n".join(cached_parts),
    max_lines=60,
)

In [ ]:
# ── 2b: Per-entity prompt (sample: waste_bin) ────────────────────────────────
topo_order = P.topological_order(ENTITIES, EDGES)
print("Topological order:", topo_order)

prompt_s2 = P.build_state2_entity_prompt(
    causal_data=CAUSAL_DATA,
    entity_id="waste_bin",
    entity_obj=SAMPLE_ENTITY_OBJ,
    accumulator_blob="",           # empty — first entity
    interface_digest={"classes": []},
    policy_outline=POLICIES,
    selected_metrics=METRICS,
    omit_cached_context=False,     # inline mode (no Gemini cache)
)

show("State 2 — Assembled Prompt (entity: waste_bin, inline mode)", prompt_s2)

In [ ]:
# ── 2c: Same prompt with omit_cached_context=True (cache-hit mode) ───────────
prompt_s2_cached = P.build_state2_entity_prompt(
    causal_data=CAUSAL_DATA,
    entity_id="waste_bin",
    entity_obj=SAMPLE_ENTITY_OBJ,
    accumulator_blob="",
    interface_digest={"classes": []},
    policy_outline=POLICIES,
    selected_metrics=METRICS,
    omit_cached_context=True,      # stable prefix supplied via Gemini cache
)

show("State 2 — Assembled Prompt (entity: waste_bin, cache-hit mode)", prompt_s2_cached)

---
## State 3 — Environment Code
**Template injected**: `environment_template.py`  
**Input**: all generated entity code (accumulator blob) + policy outline + map graph  
**Output**: Python class subclassing `SimulationEnvironment`

In [ ]:
# Fake accumulator blob: stub entity code representing prior state2 output
STUB_ENTITY_CODE = """\
# === FILE: waste_bin.py ===
from entity_object_template import entity_object

class Entity_WasteBin(entity_object):
    def __init__(self, entity_object_id: str):
        super().__init__(entity_object_id)
        self.fill_level: float = 0.0
        self.capacity: float = 100.0

    def step(self, dt: float, env) -> None:
        pass

    def empty(self) -> None:
        self.fill_level = 0.0

    def on_query(self, metric_name: str) -> dict:
        if metric_name == 'bin_fill_level':
            return {'fill_level': self.fill_level}
        return {}
"""

SAMPLE_MAP_GRAPH = {
    "vertices": [
        {"id": "building_a", "label": "Building A", "type": "building", "x": 0, "y": 0},
        {"id": "depot",      "label": "Waste Depot",  "type": "depot",    "x": 100, "y": 0},
    ],
    "edges": [
        {"id": "e1", "source": "building_a", "target": "depot", "label": "road", "weight": 1.0}
    ]
}

prompt_s3 = P.build_state3_environment_prompt(
    entities_blob=STUB_ENTITY_CODE,
    policy_outline=POLICIES,
    map_graph=SAMPLE_MAP_GRAPH,
)

show("State 3 — Assembled Prompt (with map graph)", prompt_s3)

In [ ]:
# ── State 3 without map (map_graph=None) ─────────────────────────────────────
prompt_s3_nomap = P.build_state3_environment_prompt(
    entities_blob=STUB_ENTITY_CODE,
    policy_outline=POLICIES,
    map_graph=None,
)

show("State 3 — Assembled Prompt (no map)", prompt_s3_nomap)

---
## State 4 — Policy Code
**Template injected**: `policy_template.py`  
**Iterative**: one call per policy rule  
**Output**: Python class subclassing `Policy`

In [ ]:
STUB_ENV_CODE = """\
from environment_template import SimulationEnvironment

class GarbageSimulationEnvironment(SimulationEnvironment):
    def __init__(self, entities=None, policies=None):
        super().__init__(entities=entities, policies=policies)
"""

prompt_s4 = P.build_state4_policy_prompt(
    causal_data=CAUSAL_DATA,
    rule=SAMPLE_RULE,
    entities_blob=STUB_ENTITY_CODE,
    environment_code=STUB_ENV_CODE,
    policies_accumulator="",       # empty — first policy
    map_graph=SAMPLE_MAP_GRAPH,
)

show("State 4 — Assembled Prompt (rule: empty_full_bin)", prompt_s4)

---
## State 5 — Policy Judge (Pass 1 + Pass 2)
**LLM-as-Judge**: reviews generated policy for concrete bugs.  
Pass 1 → identify issues.  Pass 2 → fix them.

In [ ]:
SAMPLE_POLICY_CODE = """\
from policy_template import Policy

class Entity_EmptyFullBinPolicy(Policy):
    @property
    def policy_name(self) -> str:
        return 'empty_full_bin'

    def apply(self, entity_object, context, env=None):
        return {}

    def before_tick(self, env, dt):
        for entity in env.entities:
            if hasattr(entity, 'fill_level') and entity.fill_level >= entity.capacity:
                entity.emtpy()   # intentional typo — judge should catch this
"""

ENTITY_CODE_INDEX = {"waste_bin": STUB_ENTITY_CODE}

MAP_ACCESSOR_API = """\
env.get_nodes() -> list[dict]
env.get_node(node_id) -> dict | None
env.get_edges() -> list[dict]
env.get_neighbors(node_id) -> list[str]
env.get_node_types() -> list[str]"""

# ── Pass 1: identify bugs ────────────────────────────────────────────────────
prompt_judge_p1 = P.build_policy_judge_pass1_prompt(
    policy_code=SAMPLE_POLICY_CODE,
    rule_contract=SAMPLE_RULE,
    entity_code_index=ENTITY_CODE_INDEX,
    map_accessor_api=MAP_ACCESSOR_API,
)
show("State 5 — Judge Pass 1 Prompt (identify bugs)", prompt_judge_p1)

In [ ]:
# ── Pass 2: fix bugs ─────────────────────────────────────────────────────────
SAMPLE_ISSUES = [
    {
        "severity": "critical",
        "location": "before_tick",
        "description": "entity.emtpy() is a typo — method is entity.empty()",
        "suggested_fix": "Change entity.emtpy() to entity.empty()",
    }
]

prompt_judge_p2 = P.build_policy_judge_pass2_prompt(
    policy_code=SAMPLE_POLICY_CODE,
    rule_contract=SAMPLE_RULE,
    entity_code_index=ENTITY_CODE_INDEX,
    map_accessor_api=MAP_ACCESSOR_API,
    issues=SAMPLE_ISSUES,
)
show("State 5 — Judge Pass 2 Prompt (fix bugs)", prompt_judge_p2)

---
## Protocol Validators
Dry-run the validators against stub code to verify they work as expected.

In [ ]:
GOOD_ENTITY = """\
from entity_object_template import entity_object
class Entity_WasteBin(entity_object):
    def step(self, dt: float, env) -> None:
        pass
    def empty(self) -> None:
        self.fill_level = 0.0
"""
BAD_ENTITY = GOOD_ENTITY.replace("def step", "def _step")  # missing step

GOOD_ENV = """\
from environment_template import SimulationEnvironment
class MyEnv(SimulationEnvironment):
    def __init__(self, entities=None, policies=None):
        super().__init__(entities, policies)
    def tick(self, dt: float) -> None:
        super().tick(dt)
"""

GOOD_POLICY = """\
from policy_template import Policy
class EmptyBinPolicy(Policy):
    @property
    def policy_name(self): return 'empty_bin'
    def apply(self, entity_object, context, env=None): return {}
    def before_tick(self, env, dt):
        pass
"""

cases = [
    ("Entity GOOD",  P.validate_entity_protocol,      GOOD_ENTITY,  ["empty"]),
    ("Entity BAD",   P.validate_entity_protocol,      BAD_ENTITY,   ["empty"]),
    ("Env GOOD",     P.validate_environment_protocol, GOOD_ENV,     None),
    ("Policy GOOD",  P.validate_policy_protocol,      GOOD_POLICY,  None),
]

for label, fn, src, extra in cases:
    if extra is not None:
        errs = fn(src, required_methods=extra)
    else:
        errs = fn(src)
    status = "✅ PASS" if not errs else f"❌ FAIL: {errs}"
    print(f"{label:20s} → {status}")

---
## Prompt Size Summary
Character counts for every assembled prompt — useful for estimating token usage.

In [ ]:
named_prompts = [
    ("state1  entity list",          prompt_s1),
    ("state1b policy outline",        prompt_s1b),
    ("state1c entity deps",           prompt_s1c),
    ("state1d metrics draft",         prompt_s1d),
    ("state2  entity (inline)",       prompt_s2),
    ("state2  entity (cache-hit)",    prompt_s2_cached),
    ("state2  cached context",        "\n\n".join(cached_parts)),
    ("state3  env (with map)",        prompt_s3),
    ("state3  env (no map)",          prompt_s3_nomap),
    ("state4  policy",                prompt_s4),
    ("state5  judge pass1",           prompt_judge_p1),
    ("state5  judge pass2",           prompt_judge_p2),
]

print(f"{'Stage':<32} {'chars':>8}  {'~tokens':>8}")
print("-" * 54)
for name, text in named_prompts:
    chars = len(text)
    approx_tokens = chars // 4
    print(f"{name:<32} {chars:>8,}  {approx_tokens:>8,}")